In [2]:
import torch
from master_model import MasterModel
from ttokenizer import Tokenizer

u_tokenizer= Tokenizer("tokenizer.json")

prompt = "the capital of united"
tokens = u_tokenizer.encode(prompt) 
tokens

tensor([ 0, 61,  1, 61,  2, 61,  3])

In [6]:
torch.manual_seed(42)   
u_model = MasterModel(
    vocab_size=len(u_tokenizer.vocab),
    embedding_dim=4,
    num_heads=2,
    context_length=32,
    num_layers=3
)
out = u_model(tokens)
out.shape

torch.Size([7, 64])

In [8]:
out[0]

tensor([-0.2160,  0.8183,  0.7515, -0.4401, -0.1353, -0.7315,  0.8828,  0.3908,
        -0.6068,  0.8389,  0.5563, -0.2742, -0.5911,  0.4069,  0.0076,  0.8122,
         0.0261,  0.2976, -0.6606,  0.2326, -0.1282, -0.0855,  0.1598,  0.5755,
        -0.6182,  0.5031, -0.0223, -0.1538,  0.0774,  0.5979,  0.1850, -0.4517,
         0.4368,  0.6212,  0.5900,  0.2398, -0.3362,  0.3950,  0.1615,  0.2315,
         0.2429, -0.4570,  0.0023,  0.1142,  0.3419,  0.2118,  0.1077,  1.2604,
         0.0869, -0.9315,  0.2965,  0.1474, -0.4400,  1.1339,  0.0053, -0.3927,
         0.6275, -0.3447,  1.1692,  0.1584, -0.0087, -0.1225, -0.3083, -0.5746],
       grad_fn=<SelectBackward0>)

In [5]:
u_model

MasterModel(
  (embedding): Embedding(64, 4)
  (pos_embedding): Embedding(32, 4)
  (layers): Sequential(
    (0): DecoderBlock(
      (self_attention): MultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=4, out_features=4, bias=True)
        )
        (projection): Linear(in_features=4, out_features=4, bias=True)
      )
      (norm1): LayerNorm()
      (mlp): MLP(
        (gate_proj): Linear(in_features=4, out_features=4, bias=True)
        (up_proj): Linear(in_features=4, out_features=4, bias=True)
        (down_proj): Linear(in_features=4, out_features=4, bias=True)
        (gelu): GELU()
      )
      (norm2): LayerNorm()
    )
    (1): DecoderBlock(
      (self_attention): MultiHeadAttention(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=4, out_features=4, bias=True)
        )
        (projection): Linear(in_features=4, out_f

In [14]:
import torch
probs = torch.softmax(out[0], dim=-1)
max_prob, max_index = torch.max(probs, dim=-1)
max_prob, max_index

(tensor(0.0433, grad_fn=<MaxBackward0>), tensor(47))

In [16]:
torch.softmax(probs, dim=-1)

tensor([0.0155, 0.0158, 0.0158, 0.0155, 0.0155, 0.0155, 0.0158, 0.0157, 0.0155,
        0.0158, 0.0157, 0.0155, 0.0155, 0.0157, 0.0156, 0.0158, 0.0156, 0.0156,
        0.0155, 0.0156, 0.0155, 0.0156, 0.0156, 0.0157, 0.0155, 0.0157, 0.0156,
        0.0155, 0.0156, 0.0157, 0.0156, 0.0155, 0.0157, 0.0157, 0.0157, 0.0156,
        0.0155, 0.0157, 0.0156, 0.0156, 0.0156, 0.0155, 0.0156, 0.0156, 0.0157,
        0.0156, 0.0156, 0.0161, 0.0156, 0.0155, 0.0156, 0.0156, 0.0155, 0.0160,
        0.0156, 0.0155, 0.0157, 0.0155, 0.0160, 0.0156, 0.0156, 0.0156, 0.0155,
        0.0155], grad_fn=<SoftmaxBackward0>)

In [17]:
probs = torch.softmax(out[-1], dim=-1)
max_prob, max_index = torch.max(probs, dim=-1)
max_prob, max_index

(tensor(0.0565, grad_fn=<MaxBackward0>), tensor(53))

In [18]:
from transformers import AutoTokenizer, AutoModelForCausalLM

q_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
q_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B")


/Users/gonul/llm-from-scratch/.llm_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 311/311 [00:00<00:00, 5239.03it/s]


In [22]:
prompt = "the capital of united"
q_tokens = q_tokenizer.encode(prompt, return_tensors="pt")
q_tokens

tensor([[ 1782,  6722,   315, 28192]])

In [23]:
q_tokenizer.tokenize(prompt)

['the', 'Ġcapital', 'Ġof', 'Ġunited']

In [24]:
q_out = q_model(q_tokens)
q_out

CausalLMOutputWithPast(loss=None, logits=tensor([[[ 4.3125,  4.5000,  3.3125,  ...,  0.9023,  0.9023,  0.9023],
         [ 6.2812,  9.1250,  2.0781,  ..., -2.9375, -2.9375, -2.9375],
         [ 4.8438,  6.8438,  1.7969,  ..., -2.7969, -2.7969, -2.7969],
         [ 5.1875,  7.3125,  3.2344,  ..., -2.9844, -2.9844, -2.9844]]],
       dtype=torch.bfloat16, grad_fn=<UnsafeViewBackward0>), past_key_values=DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer]), hidden_states=None, attentions=None)

In [33]:
q_out.logits.shape

torch.Size([1, 4, 151936])

In [26]:
q_out.logits[0, -1, :].shape

torch.Size([151936])

In [37]:
probs = torch.softmax(q_out.logits[0, 3, :], dim=-1)
max_prob, max_index = torch.max(probs, dim=-1)
max_prob, max_index, probs

(tensor(0.6094, dtype=torch.bfloat16, grad_fn=<MaxBackward0>),
 tensor(5302),
 tensor([6.9290e-07, 5.7817e-06, 9.8255e-08,  ..., 1.9554e-10, 1.9554e-10,
         1.9554e-10], dtype=torch.bfloat16, grad_fn=<SoftmaxBackward0>))

In [29]:
q_tokenizer.decode(max_index)

' states'

In [32]:
q_out.logits[0, -1, :]  

tensor([ 5.1875,  7.3125,  3.2344,  ..., -2.9844, -2.9844, -2.9844],
       dtype=torch.bfloat16, grad_fn=<SelectBackward0>)

In [ ]:
# buyuk dil modelleri siradaki tokeni tahmin eder dogru degil, aslinda model dönüşüm/transformer yapar.
#input 
#[ 1782,  6722,   315, 28192]
# output
#[?, ?, ?, 5302] -> [38297, 315, 279, 5302]
# expected output: [6722, 315, 28192, 5302]

In [38]:
# gemma modeli, transformer yapar, ve siradaki tokeni tahmin eder.
from transformers import AutoTokenizer, AutoModelForCausalLM, Gemma3ForCausalLM

gemma_tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-pt")
gemma_model = Gemma3ForCausalLM.from_pretrained("google/gemma-3-1b-pt")

Loading weights: 100%|██████████| 340/340 [00:00<00:00, 5772.30it/s]


In [55]:
g_tokens = gemma_tokenizer.encode(prompt, return_tensors="pt")
g_tokens

tensor([[    2,  1437,  5279,   529, 26974]])

In [ ]:
# input: [    2,  1437,  5279,   529, 26974]
# expected output: [ 1437,  5279,   529, 26974,  ?]

In [57]:
g_out = gemma_model(g_tokens)
g_out.logits.shape

torch.Size([1, 5, 262144])

In [58]:
probs = torch.softmax(g_out.logits[0, 0, :], dim=-1)
max_prob, max_index = torch.max(probs, dim=-1)
max_prob, max_index, probs

(tensor(0.1055, dtype=torch.bfloat16, grad_fn=<MaxBackward0>),
 tensor(184),
 tensor([2.1684e-17, 1.6671e-07, 9.4587e-10,  ..., 2.1684e-17, 1.9082e-17,
         2.1684e-17], dtype=torch.bfloat16, grad_fn=<SoftmaxBackward0>))

In [59]:
probs = torch.softmax(g_out.logits[0, 1, :], dim=-1)
max_prob, max_index = torch.max(probs, dim=-1)
max_prob, max_index, probs

(tensor(0.0972, dtype=torch.bfloat16, grad_fn=<MaxBackward0>),
 tensor(236743),
 tensor([2.1250e-17, 5.5297e-09, 1.4785e-08,  ..., 1.8757e-17, 1.8757e-17,
         2.4069e-17], dtype=torch.bfloat16, grad_fn=<SoftmaxBackward0>))

In [60]:
probs = torch.softmax(q_out.logits[0, 2, :], dim=-1)
max_prob, max_index = torch.max(probs, dim=-1)
max_prob, max_index, probs

(tensor(0.3633, dtype=torch.bfloat16, grad_fn=<MaxBackward0>),
 tensor(279),
 tensor([7.0930e-06, 5.2452e-05, 3.3714e-07,  ..., 3.4051e-09, 3.4051e-09,
         3.4051e-09], dtype=torch.bfloat16, grad_fn=<SoftmaxBackward0>))

In [61]:
probs = torch.softmax(g_out.logits[0, 3, :], dim=-1)
max_prob, max_index = torch.max(probs, dim=-1)
max_prob, max_index, probs

(tensor(0.4082, dtype=torch.bfloat16, grad_fn=<MaxBackward0>),
 tensor(506),
 tensor([3.0791e-17, 2.6822e-07, 1.0041e-09,  ..., 3.0791e-17, 2.3961e-17,
         3.0791e-17], dtype=torch.bfloat16, grad_fn=<SoftmaxBackward0>))

In [62]:
probs = torch.softmax(g_out.logits[0, 4, :], dim=-1)
max_prob, max_index = torch.max(probs, dim=-1)
max_prob, max_index, probs

(tensor(0.3066, dtype=torch.bfloat16, grad_fn=<MaxBackward0>),
 tensor(5022),
 tensor([8.0665e-17, 7.3761e-07, 1.5207e-09,  ..., 8.0665e-17, 7.1124e-17,
         9.1507e-17], dtype=torch.bfloat16, grad_fn=<SoftmaxBackward0>))

In [ ]:
# output : [38297/184, 236743, 279,506, 5022]

# input: [    2,  1437,  5279,   529, 26974]
# expected output: [ 1437,  5279,   529, 26974,  ?]

In [48]:
gemma_tokenizer.decode([38297, 236743, 279,506, 5022])

' lei ) the states'

In [63]:
gemma_tokenizer.decode([184, 236743, 279,506, 5022])

'<h1> ) the states'

In [64]:
gemma_model.generate(g_tokens, max_new_tokens=1)

tensor([[    2,  1437,  5279,   529, 26974, 41783]])

In [53]:
gemma_tokenizer.encode(" states")

[2, 5022]